# scrape judul dan url


In [13]:
import requests
import re
import json
import html as html_lib
import pandas as pd

data = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for page_num in range(1, 4):

    if page_num == 1:
        url = "https://radarmalang.jawapos.com/pendidikan"
    else:
        url = f"https://radarmalang.jawapos.com/pendidikan?page={page_num}"

    print(f"Scraping page {page_num}")

    html = requests.get(
        url,
        headers=headers
    ).text

    match = re.search(
        r'data-page="(.*?)"',
        html,
        re.DOTALL
    )

    if not match:
        print("data-page not found")
        continue

    json_str = html_lib.unescape(
        match.group(1)
    )

    page_data = json.loads(
        json_str
    )

    articles = (
        page_data["props"]
        ["category"]
        ["articles"]
        ["data"]
    )

    for article in articles:

        article_url = (
            "https://radarmalang.jawapos.com/pendidikan/"
            f"{article['article_id']}/"
            f"{article['slug']}"
        )

        data.append({
            "title": article["title"],
            "url": article_url,
            "published_date": article["date"]
        })

df = (
    pd.DataFrame(data)
    .drop_duplicates()
    .reset_index(drop=True)
)

print(df.head())

print(f"Found {len(df)} articles")

df.to_csv(
    "radar_malang_pendidikan.csv",
    index=False
)

Scraping page 1
Scraping page 2
Scraping page 3
                                               title  \
0  Catat, Jadwal Jalur Prestasi Non-Akademik SPMB...   
1    SD dan TK di Kota Malang Buka SPMB Offline u...   
2  Opsi Daftar di Jalur Afirmasi SMP Negeri Ditam...   
3  Gandeng Kemenag, Jawa Pos Radar Malang Gelar D...   
4  Segera Daftar, Diklat Jurnalistik untuk Guru S...   

                                                 url       published_date  
0  https://radarmalang.jawapos.com/pendidikan/260...  2026-06-17 15:05:22  
1  https://radarmalang.jawapos.com/pendidikan/260...  2026-06-17 14:23:05  
2  https://radarmalang.jawapos.com/pendidikan/260...  2026-06-17 14:02:58  
3  https://radarmalang.jawapos.com/pendidikan/260...  2026-06-17 09:42:33  
4  https://radarmalang.jawapos.com/pendidikan/260...  2026-06-17 09:30:22  
Found 30 articles


In [14]:
import requests
from bs4 import BeautifulSoup

url = "https://radarmalang.jawapos.com/pendidikan/2606170032/gandeng-kemenag-jawa-pos-radar-malang-gelar-diklat-jurnalistik-untuk-laz-se-malang-raya"

html = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
).text

soup = BeautifulSoup(html, "html.parser")

print(soup.get_text()[:5000])



























Gandeng Kemenag, Jawa Pos Radar Malang Gelar Diklat Jurnalistik untuk LAZ Se-Malang Raya
















































































 

 





# scrape isi url 

In [20]:
import requests
import pandas as pd
import re
import json
import html as html_lib
from bs4 import BeautifulSoup

urls_df = pd.read_csv(
    "radar_malang_pendidikan.csv"
)

data = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for idx, row in urls_df.iterrows():

    url = row["url"]

    try:

        html = requests.get(
            url,
            headers=headers,
            timeout=30
        ).text

        match = re.search(
            r'data-page="(.*?)"',
            html,
            re.DOTALL
        )

        if not match:
            print(f"[{idx+1}] data-page not found")
            continue

        page_data = json.loads(
            html_lib.unescape(
                match.group(1)
            )
        )

        article = page_data["props"]["article"]

        # Author
        author = None

        if article.get("authors"):

            authors = article["authors"]

            if isinstance(authors, list) and len(authors) > 0:

                if isinstance(authors[0], dict):
                    author = authors[0].get("name")

        # Editor
        editor = None

        if article.get("editor"):

            if isinstance(article["editor"], dict):
                editor = article["editor"].get("name")

        # Content HTML -> Text
        soup = BeautifulSoup(
            article["content"],
            "html.parser"
        )

        # Hapus gambar dan caption
        for tag in soup.find_all([
            "figure",
            "figcaption",
            "img"
        ]):
            tag.decompose()

        content = soup.get_text(
            separator="\n",
            strip=True
        )

        data.append({
            "title": article.get("title"),
            "content": content,
            "author": author,
            "editor": editor,
            "published_date": article.get("date"),
            "category": article["category"].get("name"),
            "image_url": article.get("image"),
            "url": url,
            "source": "radar_malang"
        })

        print(
            f"[{idx+1}/{len(urls_df)}] success"
        )

    except Exception as e:

        print(
            f"[{idx+1}/{len(urls_df)}] failed"
        )

        print(e)

df = pd.DataFrame(data)

df.to_csv(
    "radar_malang_articles.csv",
    index=False
)

print(
    f"\nSaved {len(df)} articles"
)

print(df.head())

[1/30] success
[2/30] success
[3/30] success
[4/30] success
[5/30] success
[6/30] success
[7/30] success
[8/30] success
[9/30] success
[10/30] success
[11/30] success
[12/30] success
[13/30] success
[14/30] success
[15/30] success
[16/30] success
[17/30] success
[18/30] success
[19/30] success
[20/30] success
[21/30] success
[22/30] success
[23/30] success
[24/30] success
[25/30] success
[26/30] success
[27/30] success
[28/30] success
[29/30] success
[30/30] success

Saved 30 articles
                                               title  \
0  Catat, Jadwal Jalur Prestasi Non-Akademik SPMB...   
1    SD dan TK di Kota Malang Buka SPMB Offline u...   
2  Opsi Daftar di Jalur Afirmasi SMP Negeri Ditam...   
3  Gandeng Kemenag, Jawa Pos Radar Malang Gelar D...   
4  Segera Daftar, Diklat Jurnalistik untuk Guru S...   

                                             content               author  \
0  MALANG KOTA, RADAR MALANG - Sesuai jadwal, pen...        Nabila Amelia   
1  MALANG KOTA\n-\n